In [8]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

while True:
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"{tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(1)

producer.flush()
producer.close()

Overwriting producer.py


In [9]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    tx = message.value

    if tx['amount'] > 3000:
        print(f"ALERT: {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']}")
# TWÓJ KOD
# Dla każdej wiadomości: sprawdź czy amount > 3000, jeśli tak — wypisz ALERT
# Format: ALERT: TX0042 | 2345.67 PLN | Warszawa | elektronika

Overwriting consumer_filter.py


In [10]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='enrich_group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Wzbogacanie transakcji o risk_level...")

for message in consumer:
    tx = message.value

    if tx['amount'] > 3000:
        tx['risk_level'] = "HIGH"
    elif tx['amount'] > 1000:
        tx['risk_level'] = "MEDIUM"
    else:
        tx['risk_level'] = "LOW"

    print(tx)
    
# TWÓJ KOD
# Czytaj z 'transactions' (użyj INNEGO group_id!)
# Dodaj pole risk_level na podstawie amount
# Wypisz wzbogaconą transakcję

Overwriting consumer_enrich.py


In [11]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0


for message in consumer:
    tx = message.value
    store = tx['store']
    amount = tx['amount']

    store_counts[store] += 1

    if store not in total_amount:
        total_amount[store] = 0
    total_amount[store] += amount

    msg_count += 1

    if msg_count % 10 == 0:
        print("\n--- PODSUMOWANIE ---")
        print("Sklep | Liczba | Suma | Średnia")

        for s in store_counts:
            count = store_counts[s]
            total = total_amount[s]
            avg = total / count if count > 0 else 0

            print(f"{s} | {count} | {total:.2f} | {avg:.2f}")

# TWÓJ KOD
# Dla każdej wiadomości:
#   1. Zwiększ store_counts[store]
#   2. Dodaj amount do total_amount[store]
#   3. Co 10 wiadomości wypisz tabelę:
#      Sklep | Liczba | Suma | Średnia

Overwriting consumer_count.py


In [12]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='stats-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# struktura: {kategoria: {count, total, min, max}}
stats = defaultdict(lambda: {
    'count': 0,
    'total': 0.0,
    'min': float('inf'),
    'max': float('-inf')
})

msg_count = 0

print("Zbieranie statystyk per kategoria...")

for message in consumer:
    tx = message.value
    category = tx['category']
    amount = tx['amount']

    # aktualizacja statystyk
    stats[category]['count'] += 1
    stats[category]['total'] += amount
    stats[category]['min'] = min(stats[category]['min'], amount)
    stats[category]['max'] = max(stats[category]['max'], amount)

    msg_count += 1

    # co 10 wiadomości podsumowanie
    if msg_count % 10 == 0:
        print("\n--- STATYSTYKI ---")
        print("Kategoria | Liczba | Suma | Min | Max")

        for cat, data in stats.items():
            print(f"{cat} | {data['count']} | {data['total']:.2f} | {data['min']:.2f} | {data['max']:.2f}")

# TWÓJ KOD

Overwriting consumer_stats.py
